Implementing a transformer

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import urllib.request
import os 
import re

torch.manual_seed(42)

url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

# Inspecting the corpus 
print(type(raw_text))
print(len(raw_text))
print(raw_text[:2000])

<class 'str'>
551846
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di nostra vita
mi ritrovai per una selva oscura
ché la diritta via era smarrita.
  Ahi quanto a dir qual era è cosa dura
esta selva selvaggia e aspra e forte
che nel pensier rinova la paura!
  Tant'è amara che poco è più morte;
ma per trattar del ben ch'i' vi trovai,
dirò de l'altre cose ch'i' v'ho scorte.
  Io non so ben ridir com'i' v'intrai,
tant'era pien di sonno a quel punto
che la verace via abbandonai.
  Ma poi ch'i' fui al piè d'un colle giunto,
là dove terminava quella valle
che m'avea di paura il cor compunto,
  guardai in alto, e vidi le sue spalle
vestite già de' raggi del pianeta
che mena dritto altrui per ogne calle.
  Allor fu la paura un poco queta
che nel lago del cor m'era durata
la notte ch'i' passai con tanta pieta.
  E come quei che con lena affannata
uscito fuor del pelago a la riva
si volge a l'acqua perigliosa e guata,
  così l'animo mio, ch'ancor fuggi

In [2]:
# Canto headers, e.g. "Inferno: Canto I", "Purgatorio: Canto XXXIII"
canto_header_re = re.compile(r'^(Inferno|Purgatorio|Paradiso):\s*Canto\s+[IVXLCDM]+\s*$')

# Front-matter title lines
title_lines = {
    "LA DIVINA COMMEDIA",
    "di Dante Alighieri",
    "INFERNO",
    "PURGATORIO",
    "PARADISO",
}

def clean_editorial_lines(text):
    cleaned = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in title_lines:
            continue
        if canto_header_re.match(stripped):
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)


raw_text = clean_editorial_lines(raw_text)
print(len(raw_text))  # sanity check: should be a bit shorter than 551846

534889


### 1- Preparing data


In [3]:
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
print(vocab_size)


67


In [4]:
string_to_index = {char: index  for index, char in enumerate(chars)}

index_to_string = { index: char for char, index in string_to_index.items()}

# Converting the whole corpus into a 1d tensor
data = torch.tensor([string_to_index[char] for char in raw_text])
data.shape

torch.Size([534889])

### 2 - Data splitting
The text is one continuous narrative and shuffling individual characters (or even lines) would let a model see fragments of validation text sitting next to training text at the boundaries it's trained on. 

In [5]:
# Data splitting
n1 = int(0.9 * len(data))
n2 = int(0.95 * len(data))

train = data[:n1]
val = data[n1:n2]
test = data[n2:]


### 3 - Batching

In [6]:
def get_batch(split, batch_size, block_size):
    
    if split == 'train':
        data_split = train
    elif split == 'val':
        data_split = val
    elif split == 'test':
        data_split = test
    else:
        raise ValueError('Split must be "train", "val" or "test"')

    ix = torch.randint(0, len(data_split) - block_size, (batch_size,))

    X = torch.stack([
        data_split[index : index + block_size] for index in ix
    ])

    Y = torch.stack([
        data_split[ index +1: index + block_size+1] for index in ix
    ])

    return X, Y



### 4 - Hyperparameter

In [7]:
# Parameters

batch_size = 32  # sequences
block_size = 128 # characters per sequence


d_model = 384 #embedding dimension describing each character
n_heads = 4 # attention heads
d_head = d_model // n_heads # dimension handled by one attention head 
d_ff = 4 * d_model #  Hidden dimension of the feed-forward network

#Historically, Transformer architectures commonly used an 
# FFN intermediate dimension around 4 × d_model

xb, yb = get_batch('train', batch_size, block_size)

print(''.join(index_to_string[t.item()] for t in xb[0]))
print(''.join(index_to_string[t.item()] for t in yb[0]))
print(xb.shape, yb.shape)


tar con organi si stea;
  ch'or sì or no s'intendon le parole.




  Poi fummo dentro al soglio de la porta
che 'l mal amor de l
ar con organi si stea;
  ch'or sì or no s'intendon le parole.




  Poi fummo dentro al soglio de la porta
che 'l mal amor de l'
torch.Size([32, 128]) torch.Size([32, 128])


### 5 - Head and FFN classes

In [8]:
# Class for head of attention
class Head(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()  # required first line 
        self.W_q = nn.Linear(d_model, d_head, bias=False)
        self.W_k = nn.Linear(d_model, d_head, bias=False)
        self.W_v = nn.Linear(d_model, d_head, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C  = x.shape  # Batch-size, Tokens in the context, C embedding space
        Q = self.W_q(x)
        V = self.W_v(x)
        K = self.W_k(x)

        scores = Q @ K.transpose(-2, -1) / (self.W_q.out_features ** 0.5) # how strongly each position attend to another
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim= -1) # softmax normalised attention scores

        return weights @ V
 

class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model, d_head, n_heads):
        super().__init__()
        self.heads = nn.ModuleList([Head(d_model, d_head) for _ in range(n_heads)])
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        head_outputs = [head(x) for head in self.heads]
        out = torch.cat(head_outputs, dim= -1)
        
        return self.proj(out) 


In [9]:
 # MLP 
class FeedForward(nn.Module):
    
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model) 
        )


    def forward(self, x):
        return self.net(x)



In [10]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model, d_head, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, d_head, n_heads)
        self.ln2 = nn.LayerNorm(d_model) 
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # running multi head attention 
        x = x + self.ffn(self.ln2(x)) #  residiual connection
        return x


In [11]:
class TransformerLanguageModel(nn.Module):

    def __init__(self, vocab_size, block_size, d_model, d_head, n_heads, d_ff):
        super().__init__()

        self.block_size = block_size

        # token embeddings
        self.token_embeddings = nn.Embedding(vocab_size, d_model)

        # Position embeding
        self.pos_embeddings = nn.Embedding(block_size, d_model)

        # Transformer block 
        #self.block = TransformerBlock(d_model, d_head, n_heads, d_ff) # initial block
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, d_head, n_heads, d_ff)
            for _ in range(2)])
        # Final normalization 
        self.ln_f = nn.LayerNorm(d_model)

        # Convert hidden representation into vocabulary logits 
        self.lm_head = nn.Linear(d_model, vocab_size)


    def forward(self, idx, targets= None):
        B, T = idx.shape   # idx = Integer token IDs entering the model

        # Token embeddings 
        token_emb = self.token_embeddings(idx)

        # Position embeddings
        positions = torch.arange(T, device=idx.device)
        pos_emb = self.pos_embeddings(positions)

        # Combine token + position information
        x = token_emb + pos_emb

        # transformer block
        #x = self.block(x) # initial block
        for block in self.blocks:
           x = block(x)
           
        # final normalization
        x = self.ln_f(x)

        # vocabulary logits 
        logits = self.lm_head(x)

        loss = None
        if targets is not None:

            B,T,C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss


### 6 - Instantiating model

In [12]:
model = TransformerLanguageModel(
    vocab_size = vocab_size,
    block_size = block_size,
    d_model = d_model,
    d_head = d_head, 
    n_heads = n_heads,
    d_ff = d_ff
)

print(model)

TransformerLanguageModel(
  (token_embeddings): Embedding(67, 384)
  (pos_embeddings): Embedding(128, 384)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (W_q): Linear(in_features=384, out_features=96, bias=False)
            (W_k): Linear(in_features=384, out_features=96, bias=False)
            (W_v): Linear(in_features=384, out_features=96, bias=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (ln2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
        )
      )
    )
  )
  (ln_f): LayerNorm((384,), eps=1e

In [13]:
@torch.no_grad()

def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train','val']:
        split_losses = []
        for _ in range(50):
            xb, yb = get_batch(split, batch_size, block_size)
            _, loss = model(xb,yb)
            split_losses.append(loss.item())
        losses[split] = sum(split_losses) / len(split_losses)
    model.train()
    return losses 


optimizer = torch.optim.AdamW(model.parameters(), lr= 3e-4)
max_steps = 3000

for step in range(max_steps):
    xb, yb = get_batch('train', batch_size, block_size)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
        

step 0: train 3.9093, val 3.9114
step 300: train 2.0877, val 2.0896
step 600: train 1.8557, val 1.8808
step 900: train 1.7526, val 1.7631
step 1200: train 1.6807, val 1.6946
step 1500: train 1.6321, val 1.6554
step 1800: train 1.5823, val 1.6076
step 2100: train 1.5462, val 1.5829
step 2400: train 1.5205, val 1.5622
step 2700: train 1.4906, val 1.5548


### Sampling new characters


In [14]:
@torch.no_grad()

def generate(model, idx, max_new_tokens):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ =model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim =-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)
    model.train()
    return idx

# Generate output 
context = torch.zeros((1,1), dtype = torch.long)
generated = generate(model, context, max_new_tokens=500)
text= ''.join(index_to_string[i.item()] for i in generated[0])
print(text)


con amago, non sedere al volto
dianzi di là, cominciò è sembiano
ne vidi guardino odamente a quella..
  Così come uri
ch'I' mi fu buoppia non susto l'umici.




  Io al com'io di veggentiano al busto,
ciò priego i Clor benigno affustro;
e di retro Iorno udicio, l'ombra che,
  men furo m'accope, metta e tò latterò la sua veluce
che s'arova l'altro fattala gente;
  che del giornis e Virgilore assai.
Cesano i dietro e volte fei messe
quei crogeato e telo i formi, morte;
  ma quantula cagpagne che n


# Architecture map
## Diagram 1 — overall pipeline

idx
(32, 64)
    │
    │ + token & position embeddings
    ↓
Embeddings
(32, 64, 128)
    │
    ▼
┌─────────────────────────────────────┐
│   Transformer block  (×1 layer)     │
│   shape (32, 64, 128) throughout    │
│                                     │
│   LayerNorm                         │
│       ↓                             │
│   Multi-head attention (4 heads)    │
│       ↓ + residual                  │
│   LayerNorm                         │
│       ↓                             │
│   Feed-forward (4× d_model = 512)   │
│       ↓ + residual                  │
└─────────────────────────────────────┘
    │
    ↓
Final layer norm
(32, 64, 128)
    │
    │ linear 128 → vocab_size
    ↓
Output logits
(32, 64, vocab_size)
    │
    ↓
softmax → sample → next-character prediction

## Diagram 2 — attention internals + residual connections
x
(32, 64, 128)
│
├──────────────────────────────────────────┐  skip
↓                                           │
LayerNorm                                   │
│                                           │
↓                                           │
┌───────────────────────────────────────┐   │
│  Multi-head attention                 │   │
│  4 heads × d_head 32                  │   │
│                                       │   │
│  Q (32,64,32)  K (32,64,32)  V (32,64,32) │
│                                       │   │
│  Q·Kᵀ → (32,64,64), mask + softmax    │   │
│  attention weights · V → (32,64,32)/head│ │
│  concat heads + linear proj → (32,64,128)││
└───────────────────────────────────────┘   │
↓                                           │
+ residual ◄────────────────────────────────┘
│
├──────────────────────────────────────────┐  skip
↓                                          │
LayerNorm                                  │
│                                          │
↓                                          │
Feed-forward                               │
128 → 512 → 128                            │
│                                          │
↓                                          │
+ residual ◄───────────────────────────────┘
│
↓
output
(32, 64, 128)